In [ ]:
!pip install scikit-image
!pip install onnx onnxscript

In [ ]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFilter
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# ==========================================
# CONFIG — Change these as needed
# ==========================================
PATCH_DIR="/kaggle/input/datasets/amritnoorsingh/cities-patches/data/patches"
EPOCHS=20
BATCH_SIZE=16       # Safe for Kaggle T4 GPU
LR=0.0002
VAL_SPLIT=0.2       # 80% train, 20% val
LOG_EVERY=20        # Print every N batches
WEIGHT_PATH = "/kaggle/working/terraclear_unet_weights.pth"
ONNX_PATH   = "/kaggle/working/terraclear_unet_cpu.onnx"

# ==========================================
# 1. IRREGULAR CLOUD MASK GENERATOR
# ==========================================
def generate_cloud_mask(h, w):
    """
    Generates a realistic irregular cloud blob mask using
    random ellipses + gaussian blur — far more realistic than rectangles.
    Returns a float mask (0.0 = clear, 1.0 = fully clouded).
    """
    mask=Image.new('L', (w, h), 0)
    draw=ImageDraw.Draw(mask)

    num_blobs=random.randint(3, 8)
    for _ in range(num_blobs):
        # Random ellipse center and radius
        cx=random.randint(0, w)
        cy=random.randint(0, h)
        rx=random.randint(20, 80)
        ry=random.randint(20, 60)
        opacity=random.randint(180, 255)
        draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=opacity)

    # Blur to make edges soft and natural-looking
    mask=mask.filter(ImageFilter.GaussianBlur(radius=random.randint(8, 18)))
    mask_array=np.array(mask, dtype=np.float32)/255.0
    return mask_array

def apply_cloud_mask(tensor_img, mask_array):
    """
    Alpha-blends the cloud mask over the satellite image.
    White (1.0) = cloud, original pixel (0.0) = clear ground.
    """
    cloudy=tensor_img.clone()
    mask_tensor=torch.from_numpy(mask_array).unsqueeze(0)  # (1, H, W)
    # Blend: cloudy pixel = ground*(1-mask) + white*mask
    cloudy=(cloudy*(1.0-mask_tensor))+(mask_tensor*1.0)
    return cloudy, mask_tensor

# ==========================================
# 2. DATASET WITH TRAIN/VAL SPLIT SUPPORT
# ==========================================
class SatelliteCloudDataset(Dataset):
    def __init__(self, patch_dir, filenames):
        self.patch_dir=patch_dir
        self.filenames=filenames
        self.to_tensor=transforms.ToTensor()

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        try:
            img_path=os.path.join(self.patch_dir, self.filenames[idx])
            clear_image=Image.open(img_path).convert('RGB')
            clear_tensor=self.to_tensor(clear_image)

            _, h, w=clear_tensor.shape
            mask=generate_cloud_mask(h, w)
            cloudy_tensor, mask_tensor=apply_cloud_mask(clear_tensor, mask)

            return cloudy_tensor, clear_tensor, mask_tensor

        except Exception as e:
            print(f"[WARNING] Skipping {self.filenames[idx]}: {e}")
            return torch.zeros(3,256,256), torch.zeros(3,256,256), torch.zeros(1,256,256)

def build_dataloaders(patch_dir, val_split, batch_size):
    all_files=[f for f in os.listdir(patch_dir) if f.endswith('.png')]
    total=len(all_files)
    val_size=int(total*val_split)
    train_size=total-val_size

    # Shuffle and split filenames
    random.shuffle(all_files)
    train_files=all_files[:train_size]
    val_files=all_files[train_size:]

    train_ds=SatelliteCloudDataset(patch_dir, train_files)
    val_ds=SatelliteCloudDataset(patch_dir, val_files)

    train_loader=DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader=DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    print(f"[✓] Train patches: {train_size} | Val patches: {val_size}")
    return train_loader, val_loader

# ==========================================
# 3. U-NET ARCHITECTURE (Upgraded)
# ==========================================
class DoubleConv(nn.Module):
    """Two conv layers with BatchNorm and ReLU — the U-Net building block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class TerraClearUNet(nn.Module):
    """
    U-Net with skip connections.
    Skip connections pass encoder feature maps directly to the decoder,
    preserving fine spatial details (roads, field edges, buildings)
    that a plain autoencoder loses in the bottleneck.
    """
    def __init__(self):
        super().__init__()
        # ENCODER
        self.enc1=DoubleConv(3, 64)       # 256x256
        self.enc2=DoubleConv(64, 128)     # 128x128
        self.enc3=DoubleConv(128, 256)    # 64x64
        self.enc4=DoubleConv(256, 512)    # 32x32
        self.pool=nn.MaxPool2d(2)

        # BOTTLENECK
        self.bottleneck=DoubleConv(512, 1024)  # 16x16

        # DECODER (each receives skip connection from encoder = double channels)
        self.up4=nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4=DoubleConv(1024, 512)

        self.up3=nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3=DoubleConv(512, 256)

        self.up2=nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2=DoubleConv(256, 128)

        self.up1=nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1=DoubleConv(128, 64)

        # OUTPUT — 3 channel RGB image, values in [0,1]
        self.final=nn.Conv2d(64, 3, kernel_size=1)
        self.sigmoid=nn.Sigmoid()

    def forward(self, x):
        # Encode + save skip maps
        e1=self.enc1(x)
        e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2))
        e4=self.enc4(self.pool(e3))

        b=self.bottleneck(self.pool(e4))

        # Decode + concatenate skip maps
        d4=self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3=self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2=self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1=self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.sigmoid(self.final(d1))

# ==========================================
# 4. EVALUATION METRICS
# ==========================================
def compute_metrics(pred_tensor, target_tensor):
    """
    Computes SSIM and PSNR between predicted and ground truth images.
    Higher SSIM (max 1.0) = better structural similarity.
    Higher PSNR (dB) = lower pixel error.
    """
    pred=pred_tensor.detach().cpu().numpy()
    target=target_tensor.detach().cpu().numpy()

    batch_ssim, batch_psnr=[], []
    for i in range(pred.shape[0]):
        p=np.transpose(pred[i], (1,2,0))     # CHW -> HWC
        t=np.transpose(target[i], (1,2,0))

        s=ssim(t, p, data_range=1.0, channel_axis=2)
        ps=psnr(t, p, data_range=1.0)

        batch_ssim.append(s)
        batch_psnr.append(ps)

    return np.mean(batch_ssim), np.mean(batch_psnr)

# ==========================================
# 5. TRAINING + VALIDATION LOOP
# ==========================================
def train_and_export():
    if not os.path.exists(PATCH_DIR) or len(os.listdir(PATCH_DIR))==0:
        print("[ERROR] No patches found. Run data_pipeline.py first.")
        return

    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[-] Hardware: {device.type.upper()}")

    train_loader, val_loader=build_dataloaders(PATCH_DIR, VAL_SPLIT, BATCH_SIZE)

    model=TerraClearUNet().to(device)
    criterion=nn.L1Loss()
    optimizer=optim.Adam(model.parameters(), lr=LR)
    # Reduce LR if val loss stops improving
    scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_val_loss=float('inf')
    print(f"\n[-] Starting Training for {EPOCHS} epochs...\n")
    total_start=time.time()

    for epoch in range(EPOCHS):
        # ---- TRAIN ----
        model.train()
        train_loss=0.0
        epoch_start=time.time()

        for batch_idx, (cloudy, clear, _) in enumerate(train_loader):
            cloudy, clear=cloudy.to(device), clear.to(device)

            optimizer.zero_grad()
            output=model(cloudy)
            loss=criterion(output, clear)
            loss.backward()
            optimizer.step()

            train_loss+=loss.item()

            if batch_idx % LOG_EVERY==0:
                print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        avg_train_loss=train_loss/len(train_loader)

        # ---- VALIDATE ----
        model.eval()
        val_loss=0.0
        val_ssim_scores=[]
        val_psnr_scores=[]

        with torch.no_grad():
            for cloudy, clear, _ in val_loader:
                cloudy, clear=cloudy.to(device), clear.to(device)
                output=model(cloudy)
                loss=criterion(output, clear)
                val_loss+=loss.item()

                s, p=compute_metrics(output, clear)
                val_ssim_scores.append(s)
                val_psnr_scores.append(p)

        avg_val_loss=val_loss/len(val_loader)
        avg_ssim=np.mean(val_ssim_scores)
        avg_psnr=np.mean(val_psnr_scores)
        epoch_time=round(time.time()-epoch_start, 1)

        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{EPOCHS} Summary [{epoch_time}s]")
        print(f"  Train Loss : {avg_train_loss:.4f}")
        print(f"  Val Loss   : {avg_val_loss:.4f}")
        print(f"  Val SSIM   : {avg_ssim:.4f}  (target > 0.85)")
        print(f"  Val PSNR   : {avg_psnr:.2f} dB  (target > 28 dB)")
        print(f"{'='*60}\n")

        scheduler.step(avg_val_loss)

        # Save best model checkpoint
        if avg_val_loss < best_val_loss:
            best_val_loss=avg_val_loss
            torch.save(model.state_dict(), WEIGHT_PATH)
            print(f"[✓] New best model saved (val_loss={avg_val_loss:.4f})\n")

    total_time=round((time.time()-total_start)/60, 2)
    print(f"[✓] Training complete in {total_time} minutes.")
    print(f"[✓] Best model saved to '{WEIGHT_PATH}'")

    # ---- ONNX EXPORT ----
    print("\n[-] Exporting best model to ONNX...")
    try:
        model.load_state_dict(torch.load(WEIGHT_PATH, map_location='cpu'))
        model.eval()
        model.to('cpu')

        dummy=torch.randn(1, 3, 256, 256)
        torch.onnx.export(
            model, dummy, ONNX_PATH,
            export_params=True,
            opset_version=18,
            do_constant_folding=True,
            input_names=['cloudy_image'],
            output_names=['clear_image'],
            dynamic_axes={'cloudy_image':{0:'batch'}, 'clear_image':{0:'batch'}}
        )
        print(f"[✓] ONNX exported: {ONNX_PATH}")
    except Exception as e:
        print(f"[WARNING] ONNX export failed: {e}")
        print("Run: pip install onnx onnxscript")

# ==========================================
# 6. INFERENCE DEMO — Before / After
# ==========================================
def run_inference_demo(image_path, output_path="demo_output.png"):
    """
    Loads a single image, applies a synthetic cloud, runs the model,
    and saves a side-by-side Before / After comparison image.
    """
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model=TerraClearUNet().to(device)
    if not os.path.exists(WEIGHT_PATH):
        print(f"[ERROR] No weights found at '{WEIGHT_PATH}'. Train first.")
        return
    model.load_state_dict(torch.load(WEIGHT_PATH, map_location=device))
    model.eval()

    to_tensor=transforms.ToTensor()
    to_pil=transforms.ToPILImage()

    clear_img=Image.open(image_path).convert('RGB').resize((256,256))
    clear_tensor=to_tensor(clear_img)

    mask=generate_cloud_mask(256, 256)
    cloudy_tensor, _=apply_cloud_mask(clear_tensor, mask)

    with torch.no_grad():
        output=model(cloudy_tensor.unsqueeze(0).to(device))
    output_img=to_pil(output.squeeze(0).cpu())

    # Side-by-side: Original | Cloudy | Reconstructed
    comparison=Image.new('RGB', (256*3, 256))
    comparison.paste(clear_img, (0, 0))
    comparison.paste(to_pil(cloudy_tensor), (256, 0))
    comparison.paste(output_img, (512, 0))
    comparison.save(output_path)
    print(f"[✓] Demo saved to '{output_path}' | Layout: [Original | Cloudy | Reconstructed]")

    # Print metrics
    pred_np=np.array(output_img)/255.0
    clear_np=np.array(clear_img)/255.0
    s=ssim(clear_np, pred_np, data_range=1.0, channel_axis=2)
    p=psnr(clear_np, pred_np, data_range=1.0)
    print(f"[✓] Demo Metrics — SSIM: {s:.4f} | PSNR: {p:.2f} dB")

# ==========================================
# ENTRY POINT
# ==========================================
if __name__=="__main__":
    train_and_export()

    # After training, run a demo on any patch from your dataset
    sample_patches=[f for f in os.listdir(PATCH_DIR) if f.endswith('.png')]
    if sample_patches:
        demo_img=os.path.join(PATCH_DIR, sample_patches[0])
        run_inference_demo(demo_img, output_path="demo_output.png")

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFilter
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# ==========================================
# CONFIG
# ==========================================
PATCH_DIR="/kaggle/input/datasets/amritnoorsingh/cities-patches/data/patches"
WEIGHT_PATH="terraclear_unet_weights.pth"
OUTPUT_DIR="./demo_outputs"
NUM_DEMO_SAMPLES=6   # Number of patches to demo in the grid

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# COPY THESE EXACTLY FROM ml_pipeline_v2.py
# (Architecture + Cloud Generator must match)
# ==========================================
def generate_cloud_mask(h, w):
    mask=Image.new('L', (w, h), 0)
    draw=ImageDraw.Draw(mask)
    num_blobs=random.randint(3, 8)
    for _ in range(num_blobs):
        cx=random.randint(0, w)
        cy=random.randint(0, h)
        rx=random.randint(20, 80)
        ry=random.randint(20, 60)
        opacity=random.randint(180, 255)
        draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=opacity)
    mask=mask.filter(ImageFilter.GaussianBlur(radius=random.randint(8, 18)))
    mask_array=np.array(mask, dtype=np.float32)/255.0
    return mask_array

def apply_cloud_mask(tensor_img, mask_array):
    cloudy=tensor_img.clone()
    mask_tensor=torch.from_numpy(mask_array).unsqueeze(0)
    cloudy=(cloudy*(1.0-mask_tensor))+(mask_tensor*1.0)
    return cloudy, mask_tensor

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class TerraClearUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=DoubleConv(3, 64)
        self.enc2=DoubleConv(64, 128)
        self.enc3=DoubleConv(128, 256)
        self.enc4=DoubleConv(256, 512)
        self.pool=nn.MaxPool2d(2)
        self.bottleneck=DoubleConv(512, 1024)
        self.up4=nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4=DoubleConv(1024, 512)
        self.up3=nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3=DoubleConv(512, 256)
        self.up2=nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2=DoubleConv(256, 128)
        self.up1=nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1=DoubleConv(128, 64)
        self.final=nn.Conv2d(64, 3, kernel_size=1)
        self.sigmoid=nn.Sigmoid()

    def forward(self, x):
        e1=self.enc1(x)
        e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2))
        e4=self.enc4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3=self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2=self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1=self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.sigmoid(self.final(d1))

# ==========================================
# LOAD MODEL
# ==========================================
def load_model(weight_path):
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model=TerraClearUNet().to(device)
    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.eval()
    print(f"[✓] Model loaded on {device.type.upper()}")
    return model, device

# ==========================================
# SINGLE IMAGE INFERENCE
# ==========================================
def infer_single(model, device, img_path):
    to_tensor=transforms.ToTensor()

    clear_img=Image.open(img_path).convert('RGB').resize((256,256))
    clear_tensor=to_tensor(clear_img)

    mask=generate_cloud_mask(256, 256)
    cloudy_tensor, mask_tensor=apply_cloud_mask(clear_tensor, mask)

    with torch.no_grad():
        output=model(cloudy_tensor.unsqueeze(0).to(device))
    output_tensor=output.squeeze(0).cpu()

    # Compute metrics
    pred_np=output_tensor.permute(1,2,0).numpy()
    clear_np=clear_tensor.permute(1,2,0).numpy()
    s=ssim(clear_np, pred_np, data_range=1.0, channel_axis=2)
    p=psnr(clear_np, pred_np, data_range=1.0)

    return clear_tensor, cloudy_tensor, output_tensor, mask_tensor.squeeze(0), s, p

# ==========================================
# MAIN DEMO GRID — Submission-ready figure
# ==========================================
def generate_demo_grid(model, device, patch_dir, num_samples, output_path):
    """
    Generates a publication-quality grid showing:
    Column 1: Original (Ground Truth)
    Column 2: Cloudy Input
    Column 3: Cloud Mask
    Column 4: Reconstructed Output
    With SSIM + PSNR scores per row.
    """
    all_patches=[f for f in os.listdir(patch_dir) if f.endswith('.png')]
    selected=random.sample(all_patches, min(num_samples, len(all_patches)))

    fig=plt.figure(figsize=(20, num_samples*4))
    fig.patch.set_facecolor('#0f0f1a')

    cols=4
    rows=num_samples
    gs=gridspec.GridSpec(rows, cols, figure=fig, hspace=0.35, wspace=0.05)

    col_titles=["Original (Ground Truth)", "Cloudy Input", "Cloud Mask", "Reconstructed Output"]
    col_colors=["#4ade80", "#f87171", "#facc15", "#60a5fa"]

    for col_idx, (title, color) in enumerate(zip(col_titles, col_colors)):
        ax=fig.add_subplot(gs[0, col_idx])
        ax.set_title(title, color=color, fontsize=13, fontweight='bold', pad=10)
        ax.axis('off')

    all_ssim, all_psnr=[], []

    for row_idx, patch_name in enumerate(selected):
        img_path=os.path.join(patch_dir, patch_name)
        clear_t, cloudy_t, output_t, mask_t, s, p=infer_single(model, device, img_path)

        all_ssim.append(s)
        all_psnr.append(p)

        images=[
            clear_t.permute(1,2,0).numpy(),
            cloudy_t.permute(1,2,0).numpy(),
            mask_t.numpy(),
            output_t.permute(1,2,0).numpy()
        ]
        cmaps=[None, None, 'gray', None]

        for col_idx, (img, cmap) in enumerate(zip(images, cmaps)):
            ax=fig.add_subplot(gs[row_idx, col_idx])
            ax.imshow(np.clip(img, 0, 1), cmap=cmap)
            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_edgecolor('#333355')
                spine.set_linewidth(1.5)

            # Show metrics on reconstructed column
            if col_idx==3:
                quality="EXCELLENT" if s>0.92 else "GOOD" if s>0.85 else "FAIR"
                q_color="#4ade80" if s>0.92 else "#facc15" if s>0.85 else "#f87171"
                ax.set_xlabel(
                    f"SSIM: {s:.4f}  |  PSNR: {p:.2f} dB  |  {quality}",
                    color=q_color, fontsize=9, labelpad=6
                )

            # Show patch name on original column
            if col_idx==0:
                short_name=patch_name[:20]+"..." if len(patch_name)>20 else patch_name
                ax.set_ylabel(short_name, color='#888899', fontsize=7, rotation=0,
                             labelpad=5, ha='right', va='center')

    # Overall summary title
    avg_s=np.mean(all_ssim)
    avg_p=np.mean(all_psnr)
    fig.suptitle(
        f"TerraClear U-Net — Cloud Removal Demo\n"
        f"Avg SSIM: {avg_s:.4f}  |  Avg PSNR: {avg_p:.2f} dB  |  {num_samples} Sample Patches",
        color='white', fontsize=15, fontweight='bold', y=1.01
    )

    plt.savefig(output_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"\n[✓] Demo grid saved to '{output_path}'")
    print(f"[✓] Average SSIM: {avg_s:.4f} | Average PSNR: {avg_p:.2f} dB")
    return avg_s, avg_p

# ==========================================
# LOSS CURVE PLOT
# ==========================================
def plot_loss_curve(train_losses, val_losses, ssim_scores, psnr_scores, output_path):
    """
    Call this after training by passing your logged loss lists.
    Saves a 2x2 training metrics dashboard.
    """
    fig, axes=plt.subplots(2, 2, figsize=(14, 8))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle("TerraClear U-Net — Training Dashboard", color='white',
                 fontsize=14, fontweight='bold')

    epochs=range(1, len(train_losses)+1)
    plots=[
        (axes[0,0], epochs, train_losses, "Train Loss", "#f87171"),
        (axes[0,1], epochs, val_losses,   "Val Loss",   "#60a5fa"),
        (axes[1,0], epochs, ssim_scores,  "Val SSIM",   "#4ade80"),
        (axes[1,1], epochs, psnr_scores,  "Val PSNR (dB)", "#facc15"),
    ]

    for ax, x, y, label, color in plots:
        ax.set_facecolor('#1a1a2e')
        ax.plot(x, y, color=color, linewidth=2)
        ax.fill_between(x, y, alpha=0.15, color=color)
        ax.set_title(label, color=color, fontweight='bold')
        ax.tick_params(colors='#888899')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')
        ax.grid(color='#222244', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"[✓] Loss curve saved to '{output_path}'")

# ==========================================
# ENTRY POINT
# ==========================================
if __name__=="__main__":
    model, device=load_model(WEIGHT_PATH)

    # 1. Generate demo grid
    generate_demo_grid(
        model, device,
        patch_dir=PATCH_DIR,
        num_samples=NUM_DEMO_SAMPLES,
        output_path=os.path.join(OUTPUT_DIR, "demo_grid.png")
    )

    # 2. Plot training curves
    # Replace these lists with your actual logged values from training
    # Example: copy the loss/SSIM/PSNR values printed during training
    # train_losses=[0.0686, 0.0225, 0.0203, ...]
    # val_losses=[0.0242, 0.0323, 0.0151, ...]
    # ssim_scores=[0.9033, 0.9038, 0.9525, ...]
    # psnr_scores=[27.63, 25.92, 32.14, ...]
    # plot_loss_curve(train_losses, val_losses, ssim_scores, psnr_scores,
    #                 output_path=os.path.join(OUTPUT_DIR, "training_dashboard.png"))

    print("\n[✓] All demo outputs saved to ./demo_outputs/")
    print("[✓] Ready for hackathon submission!")

In [ ]:
model, device=load_model(WEIGHT_PATH)
generate_demo_grid(model, device, patch_dir=PATCH_DIR, num_samples=6, output_path="./demo_outputs/demo_grid.png")

In [ ]:
!pip install onnx onnxscript

In [ ]:
import torch
model = TerraClearUNet()
model.load_state_dict(torch.load('terraclear_unet_weights.pth', map_location='cpu'))
model.eval()
dummy = torch.randn(1, 3, 256, 256)
torch.onnx.export(model, dummy, 'terraclear_unet_cpu.onnx',
    opset_version=11, input_names=['cloudy_image'], output_names=['clear_image'])
print("ONNX exported!")